# 02 — Tiền xử lý offline

Phụ trách: **Thành viên 1**. Cache dùng chung cho cả hai pipeline (letterbox + chia train/val/test + nhãn YOLO). Online augmentation **không** áp dụng ở đây.

## 0) Bootstrap (local / Kaggle / Colab)

- **Local**: notebook nằm trong repo nên dùng luôn repo hiện tại (không đụng vào git của bạn).
- **Kaggle / Colab**: clone repo về `/kaggle/working`, lấy đúng nhánh rồi cài thư viện. Trên Kaggle bật **Internet: On**.
- Repo private: xem `README.md` mục 12.8 (token lưu trong Kaggle Secrets).

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "TODO_URL"             # TODO: URL repo GitHub của nhóm
REPO_DIR = "TSD-Voice-Alert"
REPO_BRANCH = "main"              # Thành viên 1: data-yolo · Thành viên 2: faster-rcnn (khi nhánh đã có trên origin)

ON_KAGGLE = Path("/kaggle/working").exists()
ON_COLAB = "google.colab" in sys.modules

def sh(cmd: str, check: bool = True) -> int:
    print(">>", cmd)
    return subprocess.run(cmd, shell=True, check=check).returncode

# 1. Tìm repo đang chứa notebook này (local)
local_root = next((p for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "ts_alert").is_dir() and (p / "configs" / "shared.yaml").exists()), None)

if local_root is not None:
    repo_path = local_root
    print("Dùng repo hiện tại:", repo_path)
else:
    # 2. Chưa có repo: clone rồi lấy đúng nhánh (chỉ làm trong môi trường cloud)
    assert REPO_URL != "TODO_URL", "Điền REPO_URL ở ô này trước khi chạy trên Kaggle/Colab."
    base = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
    repo_path = base / REPO_DIR
    if not (repo_path / ".git").exists():
        sh(f'git clone --branch {REPO_BRANCH} {REPO_URL} "{repo_path}"')
    git = f'git -C "{repo_path}"'
    sh(f"{git} fetch origin {REPO_BRANCH}")
    sh(f"{git} checkout -B {REPO_BRANCH} origin/{REPO_BRANCH}")
    sh(f"{git} log --oneline -1")

os.chdir(repo_path)
if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))
print("cwd:", Path.cwd())

# Cài thư viện: chỉ trên Kaggle/Colab. Local thì dùng môi trường đã cài bằng requirements.txt.
if ON_KAGGLE or ON_COLAB:
    sh('pip install -q -e .')

import ts_alert
print("Bootstrap OK.")

In [ ]:
RUNTIME = "configs/runtime/kaggle.yaml" if ON_KAGGLE else "configs/runtime/local.yaml"
print("Runtime:", RUNTIME)

## 1) Chạy tiền xử lý

Tương đương dòng lệnh: `python -m tools.run_preprocess --data-root <DATA_PATH> --runtime <RUNTIME>`.

In [ ]:
from ts_alert.config.resolve import resolve_config, resolve_path
from ts_alert.data.preprocess_offline import run_preprocess

cfg = resolve_config(None, RUNTIME)
meta = run_preprocess(cfg)               # letterbox + chia train/val/test + data.yaml + meta.json
meta

## 2) Kiểm tra số lượng và tỉ lệ chia

In [ ]:
import pandas as pd

processed = resolve_path(cfg["paths"]["processed_dir"])
rows = []
for split in ("train", "val", "test"):
    n_img = len(list((processed / "images" / split).glob("*")))
    n_lbl = len(list((processed / "labels" / split).glob("*.txt")))
    rows.append({"split": split, "images": n_img, "labels": n_lbl})
df = pd.DataFrame(rows)
df["ratio"] = df["images"] / df["images"].sum()
print("Cấu hình mong muốn:", cfg["data"]["split"])
df

## 3) Kiểm tra trực quan: ảnh sau letterbox + nhãn YOLO

In [ ]:
import random
import cv2
import matplotlib.pyplot as plt

size = cfg["data"]["image_size"]
imgs = sorted((processed / "images" / "train").glob("*"))
random.seed(cfg["seed"])
sample = random.sample(imgs, k=min(6, len(imgs)))

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, path in zip(axes.ravel(), sample):
    img = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
    assert img.shape[:2] == (size, size), f"{path.name}: {img.shape}"
    lbl = processed / "labels" / "train" / (path.stem + ".txt")
    for line in lbl.read_text().splitlines():
        c, cx, cy, w, h = map(float, line.split()[:5])        # nhãn YOLO: chuẩn hoá theo ảnh
        x1, y1 = int((cx - w / 2) * size), int((cy - h / 2) * size)
        x2, y2 = int((cx + w / 2) * size), int((cy + h / 2) * size)
        cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
    ax.imshow(img); ax.set_title(path.name, fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

## 4) Kiểm tra rò rỉ dữ liệu giữa các tập

In [ ]:
names = {s: {p.stem for p in (processed / "images" / s).glob("*")} for s in ("train", "val", "test")}
leak = (names["train"] & names["val"]) | (names["train"] & names["test"]) | (names["val"] & names["test"])
print("Số ảnh trùng giữa các tập:", len(leak))
assert not leak, "Có rò rỉ dữ liệu giữa train/val/test!"

## 5) Trực quan hóa phân tích tập đã tiền xử lý (Processed EDA)

Các biểu đồ đối soát chất lượng sau 6 bước tiền xử lý (phân bố sau cân bằng, kiểm tra rò rỉ 0.00%, kích thước sau letterbox).

In [ ]:
from analysis import eda
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

proc_fig_dir = eda.FIG_DIR / "processed"
if not proc_fig_dir.is_dir() or not list(proc_fig_dir.glob("*.png")):
    eda.analyze_processed_dataset(processed_root=processed)

proc_figs = sorted(proc_fig_dir.glob("*.png"))
print(f"=== Tập sau tiền xử lý (Processed EDA): {len(proc_figs)} hình ===")
for f in proc_figs:
    plt.figure(figsize=(9, 5))
    plt.imshow(mpimg.imread(f))
    plt.title(f"[Processed] {f.name}", fontsize=11, fontweight="bold")
    plt.axis("off")
    plt.show()


## 6) Chia sẻ cache cho các notebook train (Kaggle)

Trên Kaggle, `data/processed` nằm ở `/kaggle/working` và **mất khi phiên kết thúc**. Để Thành viên 1 và 2 dùng chung: **Save Version → Save & Run All (Commit)**, rồi vào Output → **New Dataset** (vd. `tsd-processed`). Sau đó ở notebook 03/04 gắn dataset đó bằng Add Input và điền đường dẫn vào `CANDIDATES`.

## 7) Nhận xét (Thành viên 1 — Quy trình Tiền xử lý Offline 6 Bước)

1. **Tỉ lệ chia và Số lượng mỗi tập**:
   - Nhóm chốt tỉ lệ phân bổ theo cụm độc lập: **Train = 2.572 ảnh (73.7%)**, **Val = 555 ảnh (15.9%)**, **Test = 363 ảnh (10.4%)**.
   - Bằng giải thuật tối ưu phân bổ cụm, tập **Train đảm bảo chứa đầy đủ 52/52 lớp biển báo** (100% độ bao phủ lớp).

2. **Chất lượng Hộp nhãn và Lọc nhiễu**:
   - Đã loại bỏ triệt để **811 tiny box (< 8px)**, giữ lại **7.797 box hợp lệ**.
   - Sau letterbox 640x640 (pad viền xám 114), các bounding box khớp hoàn toàn với vị trí và tỉ lệ thực tế của biển báo trên ảnh gốc mà không bị biến dạng méo hình học.

3. **Tăng cường nhãn thiểu số & Triệt tiêu rò rỉ**:
   - Đã sinh bổ sung **274 ảnh và 274 box** cho 40 lớp thiểu số trong tập Train bằng kỹ thuật safe crop kèm cơ chế kiểm tra dHash đối chiếu chống rò rỉ sang Val/Test.
   - Sau bước 6 đối soát và dồn tập triệt để, **tỉ lệ rò rỉ giữa toàn bộ các tập đạt chính xác 0.00%** (assert pass).
